# SWOT PIXC Lab: AOI discovery and local manifest (Phase 1)

This notebook demonstrates the Phase 1 path **search → inspect → download/cache → verified local manifest** for the NASA SWOT L2 HR PIXC Version D collection. It intentionally does not open PIXC NetCDF variables, subset pixels, mosaic tiles, apply quality control, visualize water pixels, or calculate channel widths.

## Required project-owner input

**The project owner has not yet supplied the real multiple-channel-river AOI required by AGENTS.md Section 10.** Consequently, this is workflow scaffolding rather than the required scientific demonstration. Set `OWNER_AOI` below only after the owner provides that geometry; do not treat the illustrative bounding-box syntax as a validated river site.

Run the notebook from a checkout installed with `python -m pip install -e ".[geo,notebook]"`. CMR discovery usually works anonymously. Downloading protected PO.DAAC files requires a free NASA Earthdata Login account.

In [ ]:
from pathlib import Path

from IPython.display import display

from swot_pixc_lab import (
    DEFAULT_PIXC_COLLECTION_CONCEPT_ID,
    DEFAULT_PIXC_SHORT_NAME,
    DEFAULT_PIXC_VERSION,
    LocalFilesUnavailableError,
    PixcCollection,
)

print(DEFAULT_PIXC_SHORT_NAME, DEFAULT_PIXC_VERSION)
print(DEFAULT_PIXC_COLLECTION_CONCEPT_ID)

## 1. Configure the AOI and UTC date interval

Accepted AOIs are a WGS 84 bounding box `(west, south, east, north)`, a GeoJSON Polygon/MultiPolygon, or a GeoDataFrame-like object with a declared CRS. A date-only end value includes the complete UTC day. CMR tests granule-footprint intersection; it does not clip the pixel cloud to this geometry.

In [ ]:
# Leave this as None until the project owner supplies the real AOI.
OWNER_AOI = None

# Illustrative syntax only; do not substitute it for the owner's site.
EXAMPLE_BOUNDING_BOX_SYNTAX = (-91.20, 30.00, -91.10, 30.10)

START_DATE = "2025-06-01"
END_DATE = "2025-06-07"
CACHE_DIR = Path("data/pixc")  # Run Jupyter from the repository root.

if OWNER_AOI is None:
    print("Set OWNER_AOI before running the live discovery.")

## 2. Search the pinned PO.DAAC Version D collection

The default CMR collection concept ID is `C3233944986-POCLOUD` (`SWOT_L2_HR_PIXC_D`). Pinning the concept ID prevents an implicit switch to a different product version. It does not freeze the live catalog, so preserve the returned metadata table for provenance.

In [ ]:
collection = None
if OWNER_AOI is None:
    print("Discovery skipped: the project-owner AOI is still required.")
else:
    collection = PixcCollection.search(
        aoi=OWNER_AOI,
        start_date=START_DATE,
        end_date=END_DATE,
        collection_concept_id=DEFAULT_PIXC_COLLECTION_CONCEPT_ID,
    )
    print(collection)
    if collection.search_notes:
        print("Search notes:", *collection.search_notes, sep="\n- ")

## 3. Inspect matches before any download

The metadata table preserves available CMR identifiers, observation times, cycle/pass/tile fields, product/PGE versions, URLs, size/checksum information, footprint bounds, metadata warnings, and—after caching—the local path. Missing fields remain null rather than being guessed.

In [ ]:
SUMMARY_COLUMNS = [
    "observation_datetime",
    "cycle",
    "pass",
    "tile",
    "filename",
    "product_version",
    "size_bytes",
    "metadata_warnings",
]

metadata = None
if collection is None:
    print("Nothing to inspect until OWNER_AOI is set and discovery runs.")
else:
    metadata = collection.table
    display(metadata[SUMMARY_COLUMNS])
    known_bytes = metadata["size_bytes"].sum(min_count=1)
    print(f"Unique granules: {len(metadata)}")
    print(f"Known total bytes: {known_bytes!s}")

### Review candidate cycles and passes

Use this summary to decide which observation window is scientifically relevant. In the current Phase 1 API, `download()` retrieves every granule in the searched collection. Tighten the AOI/date interval and rerun the search before opting in if the result is broader than intended.

In [ ]:
candidate_observations = None
if metadata is not None:
    candidate_observations = (
        metadata[["observation_datetime", "cycle", "pass", "tile"]]
        .drop_duplicates()
        .sort_values(
            ["observation_datetime", "cycle", "pass", "tile"],
            na_position="last",
        )
    )
    display(candidate_observations)

## 4. Opt in to full-tile download/cache

PIXC matches are full NetCDF tiles and can be large. The guard remains `False` by design. After inspecting the count and known sizes, set it to `True` deliberately. `earthaccess` uses normal Earthdata Login mechanisms; `persist_credentials=False` prevents this call from writing credentials. Existing cache files are validated, and new files are staged before they are committed.

In [ ]:
DOWNLOAD_FULL_TILES = False

if collection is None:
    print("Download skipped: run discovery with the project-owner AOI first.")
elif not DOWNLOAD_FULL_TILES:
    print("Download disabled. Inspect matches, then opt in explicitly.")
elif len(collection) == 0:
    print("The search returned no granules; there is nothing to download.")
else:
    collection.download(
        CACHE_DIR,
        auth_strategy="all",
        persist_credentials=False,
        verify="auto",
        show_progress=True,
    )
    print(f"Cached {len(collection.local_files)} file(s) in {CACHE_DIR}.")

## 5. Open the verified local manifest

Phase 1 `open()` is local-only: it never downloads and never parses NetCDF content. It validates the expected cache files and returns their paths plus the metadata/provenance table. The attempt below is safe even when downloading remains disabled; it reports which files are absent.

In [ ]:
local = None
if collection is None:
    print("Local manifest skipped: no discovery collection exists.")
else:
    try:
        local = collection.open(cache_dir=CACHE_DIR, verify="auto")
    except LocalFilesUnavailableError as exc:
        print(exc)
    else:
        print(f"Verified local files: {len(local)}")
        display(local.table)

### Optionally save the Phase 1 provenance records

The CSV records the live CMR granule metadata and resolved cache paths; the JSON records the normalized collection, time, and spatial query. Preserve both with an analysis, but remember that local paths are machine-specific.

In [ ]:
WRITE_PROVENANCE_FILES = False

if local is None:
    print("No verified local manifest is available to save.")
elif not WRITE_PROVENANCE_FILES:
    print("Provenance-file writing is disabled.")
else:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = CACHE_DIR / "granule_manifest.csv"
    local.table.to_csv(manifest_path, index=False)
    import json
    query_path = CACHE_DIR / "search_provenance.json"
    provenance = local.provenance.as_dict() if local.provenance else {}
    query_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    print(manifest_path.resolve())
    print(query_path.resolve())

## Stop: end of Phase 1

The local object above is a file manifest, not a parsed pixel-cloud dataset. Reading official Version D NetCDF groups/variables, exact AOI subsetting, tile-boundary handling, mosaicking, quality-flag interpretation, transparent QC, visualization, research-ready pixel export, and all multi-channel measurements require later phases and validation with real files and a SWOT scientist.

## Authoritative references

- [PO.DAAC SWOT L2 HR PIXC Version D collection](https://podaac.jpl.nasa.gov/dataset/SWOT_L2_HR_PIXC_D)
- [CMR directory for collection `C3233944986-POCLOUD`](https://cmr.earthdata.nasa.gov/virtual-directory/collections/C3233944986-POCLOUD)
- [SWOT PIXC Product Description Document](https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-docs/web-misc/swot_mission_docs/pdd/D-56411_SWOT_Product_Description_L2_HR_PIXC_20250224a_RevC_clean_sig_final.pdf)
- [SWOT Version D KaRIn Products Release Note](https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-docs/web-misc/swot_mission_docs/SWOT_VersionD_KaRIn_Products_Release_Note_20250423b.pdf)
- [`earthaccess` authentication guide](https://earthaccess.readthedocs.io/en/latest/user/explanation/authenticate/)
- [NASA CMR Search API](https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html)